# آزمایشگاه سری زمانی برای دانش‌آموزان

در این نوت‌بوک یک دیتاست **۱۱۰ دانش‌آموز × ۱۶ آزمون** داریم.

همه‌ی Taskهای اصلی را روی **همین داده‌ی ثابت** اجرا می‌کنیم:

1. نمایش سری زمانی  
2. Trend — روند  
3. Forecasting — پیش‌بینی  
4. Classification — طبقه‌بندی  
5. Anomaly Detection — تشخیص ناهنجاری  
6. Change-Point Detection — تشخیص نقطه‌ی تغییر  
7. Segmentation — قطعه‌بندی  
8. Clustering — خوشه‌بندی  
9. Similarity Search — جست‌وجوی الگوی مشابه

> داده‌ها مصنوعی و آموزشی‌اند و برای کلاس طراحی شده‌اند.


## ۰) بارگذاری داده

فایل `student_scores_110.csv` را در Colab آپلود کنید.

In [ ]:
import sys, subprocess, importlib.util
for package, module in [('arabic-reshaper','arabic_reshaper'), ('python-bidi','bidi')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display
def fa(text):
    return get_display(arabic_reshaper.reshape(str(text)))

from google.colab import files
uploaded = files.upload()

df = pd.read_csv("student_scores_110.csv")
df.head()


In [ ]:
exam_cols = [f"Exam{i}" for i in range(1, 17)]

print("تعداد دانش‌آموزان:", len(df))
print("تعداد آزمون‌ها:", len(exam_cols))
print("ابعاد دیتاست:", df.shape)


## ۱) یک سری زمانی را ببینیم

فقط شماره‌ی دانش‌آموز را عوض کنید.

In [ ]:
STUDENT = 0   # عددی بین 0 تا 109

student = df.iloc[STUDENT]
scores = student[exam_cols].astype(float).values

print("نام:", student["Name"])
print("کد:", student["StudentID"])
print("نمره‌ها:", scores)

plt.figure(figsize=(10,4))
plt.plot(range(1,17), scores, marker="o")
plt.xticks(range(1,17))
plt.ylim(0,20)
plt.xlabel(fa("شماره آزمون"))
plt.ylabel(fa("نمره"))
plt.title(fa("سری زمانی نمره‌ها") + " - " + fa(student["Name"]))
plt.grid(alpha=0.25)
plt.show()


## ۲) Trend — روند

یک مدل خیلی ساده برای روند، **شیب خط** است:

\[
\text{Trend}=\frac{x_{\text{last}}-x_{\text{first}}}{N-1}
\]

- مثبت → بهبود
- منفی → افت
- نزدیک صفر → تقریباً ثابت


In [ ]:
trend = (scores[-1] - scores[0]) / (len(scores)-1)

print("Trend =", round(trend, 3))

if trend > 0.15:
    print("روند کلی: صعودی")
elif trend < -0.15:
    print("روند کلی: نزولی")
else:
    print("روند کلی: تقریباً ثابت")


## ۳) Forecasting — پیش‌بینی

ساده‌ترین پیش‌بینی: میانگین سه نمره‌ی آخر

\[
\hat{x}_{t+1}=\frac{x_t+x_{t-1}+x_{t-2}}{3}
\]


In [ ]:
window = 3
last_scores = scores[-window:]
prediction = last_scores.mean()

print("سه نمره آخر:", last_scores)
print("پیش‌بینی آزمون بعدی:", round(prediction, 2))

plt.figure(figsize=(10,4))
plt.plot(range(1,17), scores, marker="o", label=fa("نمره‌های مشاهده‌شده"))
plt.scatter([17], [prediction], s=120, label=fa("پیش‌بینی"))
plt.plot([16,17], [scores[-1], prediction], linestyle="--")
plt.xticks(range(1,18))
plt.ylim(0,20)
plt.xlabel(fa("شماره آزمون"))
plt.ylabel(fa("نمره"))
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## ۴) Classification — طبقه‌بندی

اینجا به جای پیش‌بینی یک عدد، به کل رفتار یک **برچسب** می‌دهیم.

\[
\Delta=x_{\text{last}}-x_{\text{first}}
\]


In [ ]:
def classify(scores, threshold=2.0):
    change = scores[-1] - scores[0]
    if change > threshold:
        return "Improving"
    elif change < -threshold:
        return "Declining"
    else:
        return "Stable"

print("Class:", classify(scores))


In [ ]:
def row_class(row):
    s = row[exam_cols].astype(float).values
    return classify(s)

df["Class"] = df.apply(row_class, axis=1)
df[["StudentID","Name","Class"]].head(15)


In [ ]:
df["Class"].value_counts().plot(kind="bar", title=fa("طبقه‌بندی ۱۱۰ دانش‌آموز"))
plt.ylabel(fa("تعداد دانش‌آموزان"))
plt.show()


## ۵) Anomaly Detection — تشخیص ناهنجاری

از **Z-score** استفاده می‌کنیم:

\[
z=\frac{x-\mu}{\sigma}
\]

اگر \(|z|>2\) باشد، مقدار را مشکوک در نظر می‌گیریم.


In [ ]:
mean = scores.mean()
std = scores.std()

z = (scores - mean) / std
anomaly_idx = np.where(np.abs(z) > 2)[0]

print("Mean:", round(mean,2))
print("Std:", round(std,2))
print("Anomaly exams:", anomaly_idx + 1)
print("Anomaly values:", scores[anomaly_idx])

plt.figure(figsize=(10,4))
plt.plot(range(1,17), scores, marker="o")
if len(anomaly_idx):
    plt.scatter(anomaly_idx+1, scores[anomaly_idx], s=160, marker="X", label=fa("ناهنجاری"))
plt.xticks(range(1,17))
plt.ylim(0,20)
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## ۶) Change-Point Detection — تشخیص نقطه‌ی تغییر

تفاوت دو نمره‌ی متوالی:

\[
d_t=|x_t-x_{t-1}|
\]

اگر این اختلاف از یک آستانه بزرگ‌تر باشد، آن نقطه می‌تواند Change Point باشد.


In [ ]:
diff = np.abs(np.diff(scores))
threshold = 3.5

change_points = np.where(diff > threshold)[0] + 1

print("Differences:", np.round(diff,2))
print("Change points (exam numbers):", change_points + 1)

plt.figure(figsize=(10,4))
plt.plot(range(1,17), scores, marker="o")

for cp in change_points:
    plt.axvline(cp+1, linestyle="--")

plt.xticks(range(1,17))
plt.ylim(0,20)
plt.grid(alpha=0.25)
plt.show()


## ۷) Segmentation — قطعه‌بندی

اگر یک Change Point پیدا کردیم، سری را در همان نقطه به دو بخش تقسیم می‌کنیم.

In [ ]:
if len(change_points) > 0:
    cp = change_points[0]
    segment1 = scores[:cp]
    segment2 = scores[cp:]

    print("Segment 1:", segment1)
    print("Segment 2:", segment2)
    print("Mean 1:", round(segment1.mean(),2))
    print("Mean 2:", round(segment2.mean(),2))
else:
    print("Change point واضحی با این آستانه پیدا نشد.")


## ۸) Clustering — خوشه‌بندی

این بار تمام ۱۱۰ دانش‌آموز را با هم بررسی می‌کنیم.

K-Means بدون اینکه برچسب «خوب/بد» به آن بدهیم، دانش‌آموزان با الگوهای مشابه را کنار هم قرار می‌دهد.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X = df[exam_cols].astype(float).values

# برای مقایسه‌ی شکل رفتار، هر دانش‌آموز را نسبت به میانگین خودش استاندارد می‌کنیم
X_shape = X - X.mean(axis=1, keepdims=True)

model = KMeans(n_clusters=4, random_state=42, n_init=20)
df["Cluster"] = model.fit_predict(X_shape)

df[["StudentID","Name","Cluster"]].head(20)


In [ ]:
plt.figure(figsize=(11,6))

for cluster in sorted(df["Cluster"].unique()):
    member_idx = np.where(df["Cluster"].values == cluster)[0]
    mean_curve = X[member_idx].mean(axis=0)
    plt.plot(range(1,17), mean_curve, marker="o", label=fa(f"خوشه {cluster}"))

plt.xticks(range(1,17))
plt.xlabel(fa("شماره آزمون"))
plt.ylabel(fa("میانگین نمره"))
plt.title(fa("الگوی میانگین سری زمانی هر خوشه"))
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## ۹) Similarity Search — جست‌وجوی الگوی مشابه

برای یک دانش‌آموز، فاصله‌ی اقلیدسی را با همه‌ی دانش‌آموزان دیگر حساب می‌کنیم:

\[
D(A,B)=\sqrt{\sum_{t=1}^{16}(A_t-B_t)^2}
\]

فاصله‌ی کمتر → شباهت بیشتر


In [ ]:
QUERY_STUDENT = 0

query = X_shape[QUERY_STUDENT]

distances = np.sqrt(((X_shape - query)**2).sum(axis=1))
distances[QUERY_STUDENT] = np.inf

nearest = np.argsort(distances)[:5]

print("دانش‌آموز مبنا:", df.iloc[QUERY_STUDENT]["Name"])

result = df.iloc[nearest][["StudentID","Name","Class","Cluster"]].copy()
result["Distance"] = distances[nearest]
result


In [ ]:
best = nearest[0]

plt.figure(figsize=(10,4))
plt.plot(range(1,17), X[QUERY_STUDENT], marker="o",
         label=fa(df.iloc[QUERY_STUDENT]["Name"]))
plt.plot(range(1,17), X[best], marker="o",
         label=fa(df.iloc[best]["Name"]))

plt.xticks(range(1,17))
plt.ylim(0,20)
plt.xlabel(fa("شماره آزمون"))
plt.ylabel(fa("نمره"))
plt.title(fa("شبیه‌ترین دانش‌آموز"))
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# تمرین دانش‌آموز

فقط این عدد را تغییر بده:

```python
STUDENT = 25
```

و پاسخ بده:

1. روند این دانش‌آموز چیست؟
2. نمره‌ی آزمون بعدی را پیش‌بینی کن.
3. آیا Anomaly دارد؟
4. آیا Change Point دارد؟
5. در کدام Cluster قرار می‌گیرد؟
6. شبیه‌ترین دانش‌آموز به او کیست؟
